In [1]:
import sys
sys.path.insert(0, '..')   # so Python can find src/

import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    filename='../pipeline.log',
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger('lab9')

In [7]:
# load data from CSV exported in Lab 8
# If the CSV does not exist yet, run the Lab 8 pipeline first
df_raw = pd.read_csv('../data/processed/analytics/tmdb_movies.csv', low_memory=False)
print(f'Loaded {len(df_raw)} rows and {df_raw.shape[1]} columns')
df_raw.head(3)

Loaded 901 rows and 38 columns


,id,title,release_date,popularity,text,genre,release_year,director,budget_usd,revenue_usd,...,goals_against,page_scraped,vote_average,vote_count,original_language,overview,genre_ids,adult,video,original_title
0,1084242.0,Zootopia 2,2025-11-26,171.7147,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,126889.0,Alien: Covenant,2017-05-09,49.8742,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1167307.0,David,2025-12-14,62.9864,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Cell 3 - missing value report
from src.cleaning.missing_handler import report_missing

miss_report = report_missing(df_raw)
print(miss_report.to_string())

                   missing_count  missing_pct    dtype
processed_text               900        99.89      str
raw_text                     900        99.89      str
best_picture                 898        99.67   object
Total Movies                 897        99.56  float64
Total Revenue                897        99.56  float64
Average Rating               897        99.56  float64
Total Budget                 897        99.56  float64
text                         884        98.11      str
director                     861        95.56      str
revenue_usd                  861        95.56  float64
rating_imdb                  861        95.56  float64
country                      861        95.56      str
release_year                 861        95.56  float64
genre                        861        95.56      str
budget_usd                   861        95.56  float64
awards                       858        95.23  float64
nominations                  858        95.23  float64
year_scrap

In [11]:
# Save the report
miss_report.to_csv('../data/processed/cleaned/missing_report.csv')
print('Saved missing_report.csv')

Saved missing_report.csv


In [19]:
from src.cleaning.string_cleaner import (
    clean_title, clean_language_code, clean_overview_text,
    extract_year_from_release_date
)

In [20]:
df = df_raw.copy()   # always work on a copy, keep the original safe
df = clean_title(df)
df = clean_language_code(df)
df = clean_overview_text(df)
df = extract_year_from_release_date(df)

print('Columns after string cleaning:')
print(df[['title', 'original_language', 'release_year']].head(5).to_string())

Columns after string cleaning:
                            title original_language  release_year
0                      Zootopia 2               NaN          2025
1                 Alien: Covenant               NaN          2017
2                           David               NaN          2025
3  Good Luck, Have Fun, Don't Die               NaN          2026
4                        Solo Mio               NaN          2026


In [21]:
from src.analytics.regex_ops import (
    find_invalid_dates,
    find_invalid_language_codes,
    flag_short_overviews
)

In [22]:
# Find date format problems
bad_dates = find_invalid_dates(df)
print(f'Rows with bad date format: {bad_dates.sum()}')
print(df.loc[bad_dates, ['title', 'release_date']].head(5))

find_invalid_dates: 0 dates have wrong format in release_date
Rows with bad date format: 0
Empty DataFrame
Columns: [title, release_date]
Index: []


In [23]:
# Find language code problems
bad_lang = find_invalid_language_codes(df)
print(f'\nRows with bad language code: {bad_lang.sum()}')

find_invalid_language_codes: 0 invalid codes

Rows with bad language code: 0


In [24]:
# Find short overviews
short_ov = flag_short_overviews(df, min_words=5)
print(f'\nMovies with very short overviews: {short_ov.sum()}')
print(df.loc[short_ov, ['title', 'overview']].head(5))

flag_short_overviews: 661 overviews have fewer than 5 words

Movies with very short overviews: 661
                            title overview
0                      Zootopia 2      NaN
1                 Alien: Covenant      NaN
2                           David      NaN
3  Good Luck, Have Fun, Don't Die      NaN
4                        Solo Mio      NaN


In [25]:
from src.cleaning.deduplicator import (
    drop_exact_duplicates, drop_duplicate_ids, count_duplicates
)

In [26]:
print(f'Before deduplication: {len(df)} rows')

Before deduplication: 901 rows


In [27]:
# Step 1: drop completely identical rows
df = drop_exact_duplicates(df)
print(f'After exact dedup: {len(df)} rows')

After exact dedup: 329 rows


In [28]:
# Step 2: drop rows with the same TMDB id
dup_count_before = count_duplicates(df, col='id')
print(f'Duplicate ids found: {dup_count_before}')
df = drop_duplicate_ids(df, id_col='id')
print(f'After id dedup: {len(df)} rows')

Duplicate ids found: 202
After id dedup: 127 rows


In [29]:
# Cell 7 - type conversion
from src.cleaning.type_converter import (
    convert_dates, convert_numeric_columns,
    convert_category_columns, memory_report
)

In [30]:
df_before_types = df.copy()  # save snapshot for memory comparison

df = convert_dates(df)
df = convert_numeric_columns(df)
df = convert_category_columns(df)

print('\nData types after conversion:')
print(df.dtypes)


Data types after conversion:
id                            Int64
title                           str
release_date         datetime64[us]
popularity                  float32
text                            str
genre                           str
release_year                  Int64
director                        str
budget_usd                  float64
revenue_usd                 float64
rating_imdb                 float64
country                         str
Total Movies                float64
Average Rating              float64
Total Revenue               float64
Total Budget                float64
year                        float64
awards                      float64
nominations                 float64
best_picture                 object
year_scraped                float64
raw_text                        str
processed_text                  str
name                            str
wins                        float64
losses                      float64
win_pct                     float6

In [31]:
print('\nMemory comparison:')
memory_report(df_before_types, df)


Memory comparison:
Memory before: 0.11 MB
Memory after:  0.10 MB
Saved:         0.01 MB  (10.9%)


In [41]:
from src.cleaning.missing_handler import drop_rows_missing_title
from src.cleaning.validator import run_all_validations

In [42]:
df = drop_rows_missing_title(df)

In [43]:
print('Running validation on cleaned DataFrame...')
run_all_validations(df)

Running validation on cleaned DataFrame...
  PASSED: validate_no_null_titles
  PASSED: validate_vote_average_range
  PASSED: validate_release_year_range
  PASSED: validate_no_duplicate_ids
  PASSED: validate_language_codes

Validation complete: 5 passed, 0 failed


In [48]:
print('\nSummary statistics of numeric columns:')
cols_to_check = ['vote_average', 'popularity', 'vote_count', 'runtime']
existing_cols = [col for col in cols_to_check if col in df.columns]
print(df[existing_cols].describe())


Summary statistics of numeric columns:
       vote_average  popularity   vote_count
count     28.000000  116.000000         28.0
mean       6.328714  106.681015  2069.285714
std        1.517438   95.329987   6575.23659
min        0.000000   49.403000          0.0
25%        5.950000   60.161377        22.25
50%        6.600500   76.898254         78.5
75%        6.895500  107.656975        261.5
max        9.150000  828.462097      31757.0


In [49]:
print(df['title'].isna().sum())
print((df['title'].astype(str).str.strip() == '').sum())

0
0
